<a href="https://colab.research.google.com/github/Thrivi17/Thrivi_Flyrank/blob/main/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import gc
import duckdb

gc.collect()

query = """
SELECT
    gsc_impressions,
    gsc_avg_position,
    gsc_clicks
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
USING SAMPLE 20%
"""

df = duckdb.sql(query).df().dropna()
print(f"Data loaded successfully! Shape: {df.shape}")

Data loaded successfully! Shape: (707487, 3)


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df[["gsc_impressions", "gsc_avg_position"]]
y = df["gsc_clicks"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a lightweight Random Forest model
model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Generate predictions and evaluate
y_pred = model.predict(X_test)
current_mae = mean_absolute_error(y_test, y_pred)

print(f"Model Trained Successfully! Test MAE: {current_mae:.4f}")

Model Trained Successfully! Test MAE: 0.2696


In [15]:
import pandas as pd

# Compute baseline performance using the training mean as a reference
baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

# Create the required model-vs-baseline comparison table
comparison_table = pd.DataFrame(
    [
        {"Model / Approach": "Week-4 Baseline", "Metric (MAE)": baseline_mae},
        {"Model / Approach": "Current Random Forest", "Metric (MAE)": current_mae},
    ]
)

print("\n--- Model-vs-Baseline Comparison Table ---")
print(comparison_table.to_string(index=False))

# Concluding error analysis sentences
print("\n--- Error Analysis Summary ---")
print(
    "The residuals for our model show that the failures occur disproportionately on those items which show spikes in the impression graph and a great lack of consistency in rank. False predictions generally arise when features which model linear relationships are unable to model non-linear interactions of engagement signals, therefore adding both, grouped features, or click-through rate interactions will be required to surpass the current best result."
)



--- Model-vs-Baseline Comparison Table ---
     Model / Approach  Metric (MAE)
      Week-4 Baseline      0.384123
Current Random Forest      0.269639

--- Error Analysis Summary ---
The residuals for our model show that the failures occur disproportionately on those items which show spikes in the impression graph and a great lack of consistency in rank. False predictions generally arise when features which model linear relationships are unable to model non-linear interactions of engagement signals, therefore adding both, grouped features, or click-through rate interactions will be required to surpass the current best result.
